# TELEPATI — YOLO26n FAST T4×2 (Supervised)

**Tujuan:** cepat mendapatkan checkpoint, mengurangi waktu per epoch, dan tetap bisa mengejar kinerja deteksi penyakit padi 11 kelas. Tidak ada zero-shot, teacher, KD, atau oversampling berlebihan.

**Tahap:** (0) checkpoint dini: 20% train, 1 epoch, GPU0 → (1) fast transfer learning: 65% train *class-aware subset*, 4 epoch, T4×2 → (2) fine-tune full: 100% train, 6 epoch, T4×2. Semua validation dan test asli tidak berubah. Evaluasi akhir mAP50 dan Macro F1 @ IoU 0.50.

> **Penting:** Durasi dan skor tidak dijamin. Dua GPU tidak selalu lebih cepat untuk YOLO nano karena DDP overhead. Ubah `FAST_DEVICE='0'` untuk membandingkan single T4. Checkpoint otomatis disimpan per epoch di `/kaggle/working` dan **belum persisten jika runtime Kaggle benar-benar terhapus**. Untuk keamanan antar-sesi, simpan output menggunakan Kaggle Save Version atau salin checkpoint keluar dari runtime.

In [ ]:
# 1. Dependencies — notebook Kaggle perlu Internet ON pada instalasi awal / saat unduh pretrained weights
import sys, subprocess, importlib.util
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'])
else:
    print('ultralytics sudah terpasang; jika YOLO26n belum dikenali, upgrade dengan pip -U ultralytics')

In [ ]:
import gc  # membersihkan memori pada evaluasi validation/test
# 2. Imports, CUDA, dan cek T4 ×2
import os, json, time, random, re, shutil, warnings, zipfile, hashlib
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import torch
import cv2
from tqdm.auto import tqdm
from IPython.display import display
from ultralytics import YOLO
import ultralytics

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Aktifkan GPU di Kaggle Settings > Accelerator'
print('Ultralytics:',ultralytics.__version__, '| CUDA devices:',torch.cuda.device_count())
for g in range(torch.cuda.device_count()):
    print('GPU',g,':',torch.cuda.get_device_name(g))
print('CPU cores:',os.cpu_count())

In [ ]:
# 3. Konfigurasi. Gunakan dataset conversion/cache dari notebook TELEPATI lama bila tersedia.
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else Path('/content')
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/content/data')
PROJECT_ROOT = WORK_ROOT/'telepati_abf'
DATASET_ROOT = PROJECT_ROOT/'dataset_det'
RUNS_ROOT = PROJECT_ROOT/'training_runs'
EVAL_ROOT = PROJECT_ROOT/'evaluation'
EXPORT_ROOT = PROJECT_ROOT/'exports'
for p in [PROJECT_ROOT,RUNS_ROOT,EVAL_ROOT,EXPORT_ROOT]:p.mkdir(parents=True,exist_ok=True)

FORCE_REBUILD_DATASET=False
IMAGE_LINK_MODE='symlink'
RUN_FULL_LABEL_VALIDATION=False
LABEL_AUDIT_SAMPLE_PER_SPLIT=100
RUN_EXACT_DUPLICATE_AUDIT=False
FAST_DEVICE = '0,1' if torch.cuda.device_count()>=2 else '0'
# Untuk uji jika DDP dua GPU tidak lebih cepat: FAST_DEVICE = '0'

# Cocok untuk Kaggle 2× T4. Jika OOM, coba batch stage1=16 dan stage2=16.
BATCH_BOOT=16
BATCH_STAGE1=32
BATCH_STAGE2=24
RUN_BOOT=True
RUN_STAGE1=True
RUN_STAGE2=True
# Resume=True hanya untuk melanjutkan tahap sama dari last.pt (bukan untuk pindah tahap)
RESUME_BOOT=False
RESUME_STAGE1=False
RESUME_STAGE2=False

DATA_YAML=DATASET_ROOT/'data.yaml'
CLASS_ID_TO_NAME=None
print('Data:',DATASET_ROOT,'\nDevice:',FAST_DEVICE)

In [ ]:
# 4. 11 CANONICAL CLASSES & MAPPING ASLI TELEPATI
CANONICAL_NAMES = [
    "bacterial_leaf_blight",      # 0
    "bacterial_panicle_blight",   # 1
    "blast",                      # 2
    "brown_spot",                 # 3
    "false_smut",                 # 4
    "healthy",                    # 5
    "leaf_roller",                # 6
    "leaf_scald",                 # 7
    "narrow_brown_spot",          # 8
    "sheath_blight",              # 9
    "tungro",                     # 10
]

RAW_ALIASES = {
    0: ["Bacterial leaf blight", "Bacterial-leaf-blight", "Leaf-blight", "Rice-Leaf-Diseasee", "paddy"],
    1: ["Bacterial panicle Blight"],
    2: ["Blast", "Leaf blast", "Infected Blast", "leaf-Blast"],
    3: ["BrownSpot", "Brown spot", "Brown-spot"],
    4: ["False-Smut"],
    5: ["Healthy", "healthy", "Healthy Rice Leaf", "Healthy Rice beads"],
    6: ["Leaf-roller"],
    7: ["Leaf Scald", "Leaf scald", "Leaf_scald"],
    8: ["Narrow brown"],
    9: ["Sheath Blight", "SheathBlight"],
    10: ["Rice-Tungro"],
}

def normalize_label_name(name: str) -> str:
    s = str(name).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")

ALIAS_TO_CLASS = {}
for class_id, aliases in RAW_ALIASES.items():
    for alias in list(aliases) + [CANONICAL_NAMES[class_id]]:
        key = normalize_label_name(alias)
        if key in ALIAS_TO_CLASS and ALIAS_TO_CLASS[key] != class_id:
            raise ValueError(f"Alias collision: {alias}")
        ALIAS_TO_CLASS[key] = class_id

CLASS_ID_TO_NAME = dict(enumerate(CANONICAL_NAMES))

print("Canonical classes:")
for i, name in CLASS_ID_TO_NAME.items():
    print(f"  {i:2d}: {name}")


In [ ]:
# 5. COCO AUTO-DISCOVERY (train / valid / test)
# ============================================================
# 4. AUTO-DISCOVER COCO train / valid / test
# ============================================================
SPLIT_ALIASES = {
    "train": {"train", "training"},
    "valid": {"valid", "val", "validation"},
    "test": {"test", "testing"},
}

def canonical_split_name(folder_name: str):
    n = folder_name.strip().lower()
    for canonical, aliases in SPLIT_ALIASES.items():
        if n in aliases:
            return canonical
    return None

def discover_coco_splits(input_root: Path):
    annotation_files = sorted(input_root.rglob("_annotations.coco.json"))
    if not annotation_files:
        raise FileNotFoundError(
            f"Tidak menemukan '_annotations.coco.json' di {input_root}."
        )

    grouped = defaultdict(dict)
    for ann_path in annotation_files:
        split = canonical_split_name(ann_path.parent.name)
        if split:
            grouped[ann_path.parent.parent][split] = ann_path

    complete_groups = [
        (root, mapping)
        for root, mapping in grouped.items()
        if {"train", "valid", "test"}.issubset(mapping.keys())
    ]

    if complete_groups:
        complete_groups.sort(key=lambda x: (len(x[0].parts), str(x[0])))
        selected_root, selected = complete_groups[0]
        print(f"Detected dataset root: {selected_root}")
        return {k: selected[k] for k in ["train", "valid", "test"]}

    selected = {}
    for ann_path in annotation_files:
        split = canonical_split_name(ann_path.parent.name)
        if split and split not in selected:
            selected[split] = ann_path

    missing = {"train", "valid", "test"} - set(selected)
    if missing:
        print("Annotation files found:")
        for p in annotation_files:
            print(" -", p)
        raise FileNotFoundError(f"Split COCO tidak lengkap. Missing: {sorted(missing)}")

    warnings.warn(
        "train/valid/test tidak berada pada satu parent dataset yang sama. "
        "Notebook memakai kandidat pertama untuk tiap split."
    )
    return selected

COCO_JSONS = discover_coco_splits(INPUT_ROOT)
for split, ann_path in COCO_JSONS.items():
    print(f"{split:5s} -> {ann_path}")


In [ ]:
# 6. COCO->YOLO (bbox), MEMPERTAHANKAN LABEL ASLI
# ============================================================
# 5. COCO -> YOLO DETECTION CONVERTER
# - Uses COCO bbox [x, y, w, h]
# - Falls back to bbox derived from polygon if bbox is missing/invalid
# - Writes: class xc yc w h (normalized)
# ============================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def build_image_index(split_dir: Path):
    by_rel = {}
    by_name = defaultdict(list)
    for p in split_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            try:
                by_rel[p.relative_to(split_dir).as_posix()] = p
            except Exception:
                pass
            by_name[p.name].append(p)
    return by_rel, by_name

def resolve_source_image(file_name: str, split_dir: Path, by_rel, by_name):
    normalized = str(file_name).replace("\\", "/").lstrip("./")
    if normalized in by_rel:
        return by_rel[normalized]
    direct = split_dir / normalized
    if direct.exists():
        return direct
    basename = Path(normalized).name
    candidates = by_name.get(basename, [])
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        return sorted(candidates, key=lambda p: (len(p.parts), str(p)))[0]
    return None

def fast_link_image(src: Path, dst: Path):
    """Akses gambar sumber tanpa duplikasi byte. Aman untuk /kaggle/input (read-only).

    Symlink bisa lintas filesystem, hardlink hanya pada filesystem yang sama.
    Berkas yang sudah dibuat dari run terhenti tidak di-copy ulang.
    """
    if dst.exists():
        return 'existing'
    if dst.is_symlink():  # symlink rusak dari run lama
        dst.unlink()
    src_abs = src.resolve()
    if IMAGE_LINK_MODE == 'symlink':
        try:
            dst.symlink_to(src_abs)
            return 'symlink'
        except (OSError, NotImplementedError):
            pass
    try:
        os.link(src_abs, dst)
        return 'hardlink'
    except OSError:
        shutil.copyfile(src_abs, dst)
        return 'copy_fallback'

def bbox_from_segmentation(segmentation):
    if not isinstance(segmentation, list) or not segmentation:
        return None
    if segmentation and isinstance(segmentation[0], (int, float)):
        segmentation = [segmentation]
    pts = []
    for poly in segmentation:
        if isinstance(poly, (list, tuple)) and len(poly) >= 6 and len(poly) % 2 == 0:
            arr = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
            if np.isfinite(arr).all():
                pts.append(arr)
    if not pts:
        return None
    pts = np.concatenate(pts, axis=0)
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)
    return [float(x1), float(y1), float(x2-x1), float(y2-y1)]

def normalize_coco_bbox(bbox, img_w, img_h):
    if bbox is None or len(bbox) != 4 or img_w <= 0 or img_h <= 0:
        return None
    x, y, bw, bh = map(float, bbox)
    x1 = np.clip(x, 0, img_w)
    y1 = np.clip(y, 0, img_h)
    x2 = np.clip(x + bw, 0, img_w)
    y2 = np.clip(y + bh, 0, img_h)
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    wn = (x2 - x1) / img_w
    hn = (y2 - y1) / img_h
    return np.clip(np.array([xc, yc, wn, hn], dtype=np.float32), 0.0, 1.0)

def convert_coco_split(ann_path: Path, out_root: Path, split_name: str, strict_mapping=True):
    split_dir = ann_path.parent
    out_images = out_root / "images" / split_name
    out_labels = out_root / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {int(x["id"]): x for x in coco.get("images", [])}
    categories = {int(x["id"]): x for x in coco.get("categories", [])}

    category_map = {}
    unmapped = []
    for cat_id, cat in categories.items():
        raw_name = cat.get("name", "")
        key = normalize_label_name(raw_name)
        if key not in ALIAS_TO_CLASS:
            unmapped.append((cat_id, raw_name))
        else:
            category_map[cat_id] = ALIAS_TO_CLASS[key]

    if unmapped and strict_mapping:
        raise ValueError(f"Unmapped COCO categories in {ann_path}: {unmapped}")

    anns_by_image = defaultdict(list)
    for ann in coco.get("annotations", []):
        anns_by_image[int(ann["image_id"])].append(ann)

    by_rel, by_name = build_image_index(split_dir)
    stats = {
        "images_copied": 0,  # legacy key: jumlah gambar terhubung (bukan byte yang dicopy)
        "image_storage_modes": {},
        "images_missing": 0,
        "annotations_written": 0,
        "segmentation_bbox_fallbacks": 0,
        "invalid_annotations_skipped": 0,
        "class_counts": {name: 0 for name in CANONICAL_NAMES},
    }

    used_names = set()
    for image_id, meta in tqdm(images.items(), desc=f"Convert {split_name}"):
        file_name = meta.get("file_name", "")
        src = resolve_source_image(file_name, split_dir, by_rel, by_name)
        if src is None:
            stats["images_missing"] += 1
            continue

        img_w = int(meta.get("width", 0))
        img_h = int(meta.get("height", 0))
        if img_w <= 0 or img_h <= 0:
            probe = cv2.imread(str(src))
            if probe is None:
                stats["images_missing"] += 1
                continue
            img_h, img_w = probe.shape[:2]

        dst_name = Path(file_name).name or src.name
        if dst_name in used_names:
            dst_name = f"{image_id}_{dst_name}"
        used_names.add(dst_name)
        dst_img = out_images / dst_name
        storage_mode = fast_link_image(src, dst_img)
        stats["image_storage_modes"][storage_mode] = stats["image_storage_modes"].get(storage_mode, 0) + 1
        stats["images_copied"] += 1

        label_lines = []
        for ann in anns_by_image.get(image_id, []):
            cat_id = int(ann.get("category_id", -1))
            if cat_id not in category_map:
                continue
            class_id = category_map[cat_id]
            bbox = ann.get("bbox")
            norm = normalize_coco_bbox(bbox, img_w, img_h)
            if norm is None:
                bbox = bbox_from_segmentation(ann.get("segmentation"))
                norm = normalize_coco_bbox(bbox, img_w, img_h)
                if norm is not None:
                    stats["segmentation_bbox_fallbacks"] += 1
            if norm is None:
                stats["invalid_annotations_skipped"] += 1
                continue

            xc, yc, bw, bh = norm.tolist()
            label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            stats["annotations_written"] += 1
            stats["class_counts"][CANONICAL_NAMES[class_id]] += 1

        (out_labels / f"{Path(dst_name).stem}.txt").write_text(
            "\n".join(label_lines) + ("\n" if label_lines else ""),
            encoding="utf-8",
        )

    return stats

def validate_yolo_det_labels(labels_root: Path, max_files=None):
    errors = []
    checked = 0
    paths = sorted(labels_root.rglob("*.txt"))
    if max_files is not None and len(paths) > max_files:
        paths = sorted(random.Random(SEED).sample(paths, max_files))
    for label_path in tqdm(paths, desc="Audit YOLO labels", leave=False):
        for line_no, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            checked += 1
            parts = line.split()
            if len(parts) != 5:
                errors.append((str(label_path), line_no, f"expected 5 columns, got {len(parts)}"))
                continue
            try:
                cls = int(float(parts[0]))
                vals = np.asarray([float(x) for x in parts[1:]], dtype=np.float32)
            except Exception as exc:
                errors.append((str(label_path), line_no, f"parse error: {exc}"))
                continue
            if not (0 <= cls < len(CANONICAL_NAMES)):
                errors.append((str(label_path), line_no, f"class={cls}"))
            if not np.isfinite(vals).all() or (vals < 0).any() or (vals > 1).any():
                errors.append((str(label_path), line_no, "bbox outside [0,1] / non-finite"))
            if vals[2] <= 0 or vals[3] <= 0:
                errors.append((str(label_path), line_no, "non-positive bbox width/height"))
    if errors:
        raise ValueError(f"YOLO detection label validation failed. First errors: {errors[:20]}")
    print(f"YOLO labels OK: {len(paths):,} files / {checked:,} objects diperiksa"
          + (" (sample audit)" if max_files is not None else " (full audit)"))


In [ ]:
# 7. FAST AUDIT ALIAS + COCO -> YOLO DENGAN SYMLINK / CACHE
start_total = time.perf_counter()
print('Audit alias dan konversi dataset (FAST MODE)...', flush=True)
AMBIGUOUS_ALIAS_NAMES = {'paddy', 'rice_leaf_diseasee'}
for split, ann_path in COCO_JSONS.items():
    tic = time.perf_counter()
    coco = json.loads(ann_path.read_text(encoding='utf-8'))
    cat_map = {int(c['id']): str(c['name']) for c in coco['categories']}
    frequencies = Counter(cat_map.get(int(a['category_id']), '') for a in coco['annotations'])
    ambiguous = {k:v for k,v in frequencies.items() if normalize_label_name(k) in AMBIGUOUS_ALIAS_NAMES}
    if ambiguous:
        warnings.warn(f'{split.upper()}: alias ambigu masih disatukan ke bacterial_leaf_blight: {ambiguous}. '
                      'Audit dataset aslinya; mapping tetap sama seperti notebook awal agar comparable.')
    print(f'  Audit {split:<5s}: {len(coco["images"]):,} gambar, '
          f'{len(coco["annotations"]):,} anotasi ({time.perf_counter()-tic:.1f} s)', flush=True)
    del coco

DATA_YAML = DATASET_ROOT / 'data.yaml'
report_path = DATASET_ROOT / 'conversion_report.json'
if FORCE_REBUILD_DATASET and DATASET_ROOT.exists():
    print('PERINGATAN: FORCE_REBUILD_DATASET=True, menghapus konversi lokal untuk rebuild.')
    shutil.rmtree(DATASET_ROOT)
if not (DATA_YAML.exists() and report_path.exists()):
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    conversion_stats = {}
    for split in ['train','valid','test']:
        tic = time.perf_counter()
        conversion_stats[split] = convert_coco_split(
            COCO_JSONS[split], DATASET_ROOT, split_name=split, strict_mapping=True)
        print(f'  Konversi {split:<5s} selesai dalam {time.perf_counter()-tic:.1f} s | '
              f'mode: {conversion_stats[split].get("image_storage_modes", {})}', flush=True)
        # Checkpoint split, agar statistik yang berhasil tak hilang saat runtime terganggu
        (DATASET_ROOT / 'conversion_report_partial.json').write_text(
            json.dumps(conversion_stats, indent=2), encoding='utf-8')
    with report_path.open('w', encoding='utf-8') as f:
        json.dump(conversion_stats, f, indent=2)
else:
    conversion_stats = json.loads(report_path.read_text(encoding='utf-8'))
    print('CACHE HIT — hasil konversi sebelumnya dipakai tanpa menyalin ulang gambar:', DATASET_ROOT)

base_yaml = {
    'path': str(DATASET_ROOT), 'train':'images/train', 'val':'images/valid',
    'test':'images/test', 'nc':len(CANONICAL_NAMES),
    'names': {i:name for i,name in enumerate(CANONICAL_NAMES)}
}
with DATA_YAML.open('w') as f:
    yaml.safe_dump(base_yaml, f, sort_keys=False)
(DATASET_ROOT / 'class_mapping.json').write_text(json.dumps(
    {'classes':CANONICAL_NAMES, 'raw_aliases':RAW_ALIASES}, indent=2), encoding='utf-8')

max_check = None if RUN_FULL_LABEL_VALIDATION else LABEL_AUDIT_SAMPLE_PER_SPLIT
for split in ['train','valid','test']:
    validate_yolo_det_labels(DATASET_ROOT / 'labels' / split, max_files=max_check)
for split, stats in conversion_stats.items():
    print(f"{split:5s}: {stats['images_copied']:5d} images | {stats['annotations_written']:6d} objects | "
          f"missing={stats['images_missing']}")
assert all(conversion_stats[s]['images_copied'] > 0 for s in ['train','valid','test'])
print(f'FAST AUDIT + CONVERSION selesai: {time.perf_counter()-start_total:.1f} detik')


## FAST SPLITS (tanpa copy atau duplikasi gambar)

Subset disusun **hanya dari split train** dan diprioritaskan agar kelas minoritas terwakili. Setiap berkas gambar dicantumkan lewat path absolut yang tetap berada dalam folder `images/train`, sehingga Ultralytics bisa menemukan label pasangannya dalam `labels/train`.

In [ ]:
# 4. Buat subset stratifikasi-lunak: 20% untuk 1-epoch boot, 65% untuk tahap 1.
IMG_SUFFIXES={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
train_dir=DATASET_ROOT/'images'/'train'
images=sorted(p for p in train_dir.iterdir() if p.suffix.lower() in IMG_SUFFIXES and p.is_file())
assert images,'Tidak ada gambar train; jalankan cell konversi lebih dahulu'
class_sets=[]
for p in tqdm(images,desc='Indeks kelas train (sekali saja)'):
    lb=DATASET_ROOT/'labels'/'train'/(p.stem+'.txt')
    s=set()
    if lb.is_file():
        for line in lb.read_text().splitlines():
            if line.strip():
                s.add(int(float(line.split()[0])))
    class_sets.append(s)
class_sets=[set(x) for x in class_sets]
idx_by_class={c:[i for i,ss in enumerate(class_sets) if c in ss] for c in range(len(CANONICAL_NAMES))}
assert all(idx_by_class[c] for c in idx_by_class), 'Kelas kosong di training set; audit mapping/dataset'
counts={c:len(idx_by_class[c]) for c in idx_by_class}
positive=np.array([v for v in counts.values() if v>0],dtype=float)
med=float(np.median(positive))
weights=np.array([min(5.0,max([1.0]+[np.sqrt(med/max(1,counts[c])) for c in cls])) for cls in class_sets],dtype=float)

def class_aware_subset(frac, seed=42, min_images_per_class=20):
    rng=np.random.default_rng(seed)
    target=max(1,round(len(images)*frac))
    mandatory=set()
    for c,idx in sorted(idx_by_class.items(),key=lambda kv:len(kv[1])):
        # Ambil sejumlah kecil kandidat per kelas, bukan menambah duplikat image
        k=min(min_images_per_class,len(idx),max(1,target//(3*len(CANONICAL_NAMES))))
        for j in rng.choice(idx,size=k,replace=False):mandatory.add(int(j))
    if len(mandatory)>target:
        raise RuntimeError('Subset terlalu kecil untuk jaminan kelas')
    left=np.array([j for j in range(len(images)) if j not in mandatory],dtype=int)
    k=target-len(mandatory)
    if k:
        u=np.maximum(rng.random(len(left)),1e-12)
        keys=-np.log(u)/weights[left]
        chosen=left[np.argpartition(keys,k-1)[:k]] if k<len(left) else left
    else:
        chosen=np.array([],dtype=int)
    result=sorted(set(mandatory)|set(map(int,chosen)))
    assert len(result)==target
    seen=Counter(c for j in result for c in class_sets[j])
    assert all(seen[c] for c in idx_by_class)
    return result,seen

sample_root=PROJECT_ROOT/'fast_subsets'
sample_root.mkdir(exist_ok=True)
configs={}
for nm,frac in [('boot',.20),('stage1',.65)]:
    selected,seen=class_aware_subset(frac,seed=SEED)
    path=sample_root/f'{nm}_train.txt'
    # Jangan .resolve() di sini: symlink image harus tetap melewati folder images/train!
    path.write_text('\n'.join(str(images[j].absolute()) for j in selected)+'\n',encoding='utf-8')
    config={
        'path':str(DATASET_ROOT),'train':str(path),
        'val':str(DATASET_ROOT/'images'/'valid'),
        'test':str(DATASET_ROOT/'images'/'test'),
        'nc':len(CANONICAL_NAMES),'names':dict(enumerate(CANONICAL_NAMES))
    }
    cfgpath=sample_root/f'{nm}_data.yaml'
    cfgpath.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    configs[nm]=cfgpath
    print(f'{nm}: {len(selected)}/{len(images)} images, objek per kelas (image count):',dict(seen))
print('Original full training:',DATA_YAML)

## Tahap 0 · 1 epoch singkat → checkpoint awal

Memakai 20% data supervised, 448 px, **1 GPU** untuk menghindari overhead DDP pada bootstrap. Setelah epoch pertama, `last.pt` disimpan. Ini bukan model final.

In [ ]:
# 5. Runner CLI stabil untuk DDP di Kaggle (menghindari serialisasi fungsi dari notebook)
import subprocess, shutil, time
assert shutil.which('yolo'), 'CLI yolo belum tersedia; install/upgrade ultralytics'

def run_yolo_train(run_name, args, resume=False):
    run_dir=RUNS_ROOT/run_name
    last=run_dir/'weights'/'last.pt'
    if resume:
        assert last.is_file(), f'Checkpoint resume tidak ditemukan: {last}'
        command=['yolo','detect','train',f'model={last}','resume=True']
    elif last.is_file():
        print(f'[SKIP] Sudah ada checkpoint: {last}. Set RESUME_*=True jika tahap ini terputus.')
        return last
    else:
        command=['yolo','detect','train']+[f'{k}={v}' for k,v in args.items()]
    print('RUN:', ' '.join(command), flush=True)
    t=time.perf_counter()
    subprocess.run(command,check=True)
    print('Selesai dalam',round((time.perf_counter()-t)/60,2),'menit')
    assert last.exists(),f'Training berhenti tanpa checkpoint: {last}'
    return last

def base_train_args(model_path,data_path,ep,imgsz,batch,device,name,freeze=0,val=True,rect=True,lr=.001):
    return dict(
        model=str(model_path), data=str(data_path),epochs=ep,imgsz=imgsz,batch=batch,device=device,
        optimizer='AdamW',lr0=lr,lrf=.15,warmup_epochs=.5,freeze=freeze,
        workers=2,amp=True,cache=False,rect=rect,deterministic=False,seed=SEED,
        mosaic=0,mixup=0,copy_paste=0,close_mosaic=0,
        fliplr=.5,flipud=0,hsv_h=.004,hsv_s=.15,hsv_v=.12,translate=.04,scale=.12,
        val=val,patience=max(4,ep),save=True,save_period=1,plots=False,
        project=str(RUNS_ROOT),name=name,exist_ok=True,verbose=True
    )

In [ ]:
# 6. Boot: 1 epoch -> simpan last.pt sesegera mungkin
BOOT_RUN='t4x2_boot_1ep'
BOOT_LAST=RUNS_ROOT/BOOT_RUN/'weights'/'last.pt'
if RUN_BOOT:
    kwargs=base_train_args('yolo26n.pt',configs['boot'],1,448,BATCH_BOOT,'0',BOOT_RUN,freeze=6,val=False)
    BOOT_LAST=run_yolo_train(BOOT_RUN,kwargs,resume=RESUME_BOOT)
else:
    assert BOOT_LAST.is_file(),f'Boot belum tersedia: {BOOT_LAST}'
print('BOOT CHECKPOINT:',BOOT_LAST)

## Tahap 1 · 4 epoch fast transfer learning

Dataset 65% dengan keseimbangan kelas lebih baik tanpa oversampling, 448 px, `freeze=6`, dan dua T4 via DDP. `rect=True` dan tanpa mosaic mengurangi pekerjaan CPU. Pada model nano, uji juga GPU tunggal apabila dua GPU tidak lebih cepat.

In [ ]:
# 7. Fast stage 1 (checkpoint dari boot; tidak resume optimizer lintas tahap)
STAGE1_RUN='t4x2_stage1_fast'
STAGE1_LAST=RUNS_ROOT/STAGE1_RUN/'weights'/'last.pt'
if RUN_STAGE1:
    kwargs=base_train_args(BOOT_LAST,configs['stage1'],4,448,BATCH_STAGE1,FAST_DEVICE,STAGE1_RUN,freeze=6,val=False)
    STAGE1_LAST=run_yolo_train(STAGE1_RUN,kwargs,resume=RESUME_STAGE1)
else:
    assert STAGE1_LAST.exists(),f'Stage1 belum tersedia: {STAGE1_LAST}'
print('STAGE1 CHECKPOINT:',STAGE1_LAST)

## Tahap 2 · 6 epoch full-data fine-tuning

Semua gambar training asli, 576 px, backbone dibuka (`freeze=0`). Validation asli digunakan untuk memilih `best.pt`. Jika deteksi bercak kecil tidak cukup baik, naikkan resolusi 640 pada run lanjutan setelah mengevaluasi checkpoint tahap 2; jangan langsung menjalankan ratusan epoch.

In [ ]:
# 8. Fine-tuning seluruh dataset + validasi. Simpan last.pt dan epochN.pt tiap epoch.
STAGE2_RUN='t4x2_stage2_full'
STAGE2_DIR=RUNS_ROOT/STAGE2_RUN
STAGE2_LAST=STAGE2_DIR/'weights'/'last.pt'
if RUN_STAGE2:
    kwargs=base_train_args(STAGE1_LAST,DATA_YAML,6,576,BATCH_STAGE2,FAST_DEVICE,STAGE2_RUN,freeze=0,val=True,rect=False,lr=.00035)
    # Augmentasi ringan: tidak mengubah warna lesi terlalu agresif
    kwargs.update(mosaic=.10,close_mosaic=2)
    STAGE2_LAST=run_yolo_train(STAGE2_RUN,kwargs,resume=RESUME_STAGE2)
else:
    assert STAGE2_LAST.is_file(),f'Stage2 belum tersedia: {STAGE2_LAST}'
STAGE2_BEST=STAGE2_DIR/'weights'/'best.pt'
FINAL_PT=STAGE2_BEST if STAGE2_BEST.is_file() else STAGE2_LAST
print('MODEL FINAL:',FINAL_PT)

In [ ]:
# 9. Simpan salinan checkpoint + ringkasan, siap download / Save Version Kaggle
from shutil import copy2
safe=EXPORT_ROOT/'fast_t4x2'
safe.mkdir(parents=True,exist_ok=True)
for label,p in [('boot_last',BOOT_LAST),('stage1_last',STAGE1_LAST),('stage2_last',STAGE2_LAST),('final_best',FINAL_PT)]:
    if p.exists():
        out=safe/f'{label}.pt'
        if p.resolve()!=out.resolve():copy2(p,out)
        print(label,'->',out, '(',round(out.stat().st_size/1024**2,1),'MB)')
print('PENTING: /kaggle/working tidak menjamin penyimpanan lintas runtime yang hilang. Gunakan Kaggle Save Version / download file.')

## Evaluasi benchmark

mAP@50 dari validator YOLO + *custom* Macro F1 IoU 0.5 matching per kelas mengikuti notebook TELEPATI sebelumnya. Pilih confidence threshold hanya pada **validation**, kemudian gunakan threshold tersebut tetap pada **test**. Jangan menyesuaikan hyperparameter terhadap test.

In [ ]:
# 10. mAP50 di validation (split val asli)
model=YOLO(str(FINAL_PT))
metric=model.val(data=str(DATA_YAML),split='val',imgsz=640,device=0,batch=16,workers=2,plots=False,verbose=False)
print('VAL mAP50 =',round(metric.box.map50*100,2),'%')
print('VAL mAP50-95 =',round(metric.box.map*100,2),'%')

In [ ]:
MATCH_IOU=0.50
IMGSZ=640
INFER_CONF_FLOOR=0.01
PRED_BATCH=8
CONF_GRID=np.round(np.arange(.05,.61,.05),2).tolist()
# None = evaluasi seluruh validation; angka (mis. 300) = smoke test cepat saja.
EVAL_MAX_VALID_IMAGES = None
# 14. GT MATCHING METRICS (diadaptasi dari notebook TELEPATI awal)
IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
def label_path_for_image(image_path: Path, split: str):
    return DATASET_ROOT / 'labels' / split / f'{image_path.stem}.txt'
def load_gt_boxes(image_path: Path, split: str):
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)
    h, w = img.shape[:2]
    lp = label_path_for_image(image_path, split)
    boxes, classes = [], []
    if lp.exists():
        for line in lp.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, xc, yc, bw, bh = map(float, parts)
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            x2 = (xc + bw / 2) * w
            y2 = (yc + bh / 2) * h
            boxes.append([x1, y1, x2, y2])
            classes.append(int(cls))
    return {
        "boxes": np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
        "classes": np.asarray(classes, dtype=np.int64),
    }


def box_iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros((0,), dtype=np.float32)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    area1 = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area2 = np.maximum(0, boxes[:, 2]-boxes[:, 0]) * np.maximum(0, boxes[:, 3]-boxes[:, 1])
    union = area1 + area2 - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)


def evaluate_f1(records, conf_threshold, match_iou=MATCH_IOU):
    per_class = []
    total_tp = total_fp = total_fn = 0

    for class_id, class_name in CLASS_ID_TO_NAME.items():
        gt_by_image = {}
        matched_by_image = {}
        predictions = []
        total_gt = 0

        for rec_idx, rec in enumerate(records):
            gt_mask = rec["gt_classes"] == class_id
            gt_boxes = rec["gt_boxes"][gt_mask]
            gt_by_image[rec_idx] = gt_boxes
            matched_by_image[rec_idx] = np.zeros(len(gt_boxes), dtype=bool)
            total_gt += len(gt_boxes)

            pred_mask = (rec["pred_classes"] == class_id) & (rec["pred_scores"] >= conf_threshold)
            for box, score in zip(rec["pred_boxes"][pred_mask], rec["pred_scores"][pred_mask]):
                predictions.append((float(score), rec_idx, box))

        predictions.sort(key=lambda x: x[0], reverse=True)
        tp = fp = 0
        for score, rec_idx, box in predictions:
            gt_boxes = gt_by_image[rec_idx]
            if len(gt_boxes) == 0:
                fp += 1
                continue
            ious = box_iou_one_to_many(box, gt_boxes)
            ious[matched_by_image[rec_idx]] = -1
            best_idx = int(np.argmax(ious))
            if ious[best_idx] >= match_iou:
                matched_by_image[rec_idx][best_idx] = True
                tp += 1
            else:
                fp += 1

        fn = int(total_gt - tp)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        per_class.append({
            "class_id": class_id,
            "class_name": class_name,
            "gt": int(total_gt),
            "pred": int(len(predictions)),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        })
        total_tp += tp
        total_fp += fp
        total_fn += fn

    df = pd.DataFrame(per_class)
    supported = df["gt"] > 0
    macro_supported = float(df.loc[supported, "f1"].mean()) if supported.any() else 0.0
    macro_all = float(df["f1"].mean())
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (2 * micro_p * micro_r / (micro_p + micro_r)) if (micro_p + micro_r) else 0.0

    summary = {
        "conf_threshold": float(conf_threshold),
        "match_iou": float(match_iou),
        "macro_f1_supported": macro_supported,
        "macro_f1_all_classes": macro_all,
        "micro_precision": float(micro_p),
        "micro_recall": float(micro_r),
        "micro_f1": float(micro_f1),
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn),
    }
    return summary, df

In [ ]:
# 15. GENERATE DETECTOR RECORDS DAN PILIH THRESHOLD VALID
# stream=True menghindari menyimpan seluruh gambar dalam RAM.
def generate_records_for_checkpoint(checkpoint: Path, split: str, max_images=None):
    source_dir = DATASET_ROOT/'images'/split
    images = sorted(p for p in source_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    if max_images is not None:
        # represent subset via temp directory hardlink agar tetap streaming
        imgs_subset = images[:int(max_images)]
        subset = PROJECT_ROOT/'eval_subsets'/split
        if subset.exists(): shutil.rmtree(subset)
        subset.mkdir(parents=True)
        for p in imgs_subset: link_or_copy(p,subset/p.name)
        source_dir = subset
    m = YOLO(str(checkpoint))
    results = m.predict(
        source=str(source_dir),stream=True,batch=PRED_BATCH,
        imgsz=IMGSZ,conf=INFER_CONF_FLOOR,iou=0.70,max_det=300,
        device=0,verbose=False,
    )
    records=[]
    expected=len(images) if max_images is None else min(int(max_images),len(images))
    for r in tqdm(results, total=expected, desc=f'Predict {split} - {Path(checkpoint).name}'):
        img_path = Path(r.path)
        # GT read original split by stem, even if inference uses subset directory
        gt = load_gt_boxes(img_path,split)
        if r.boxes is not None and len(r.boxes)>0:
            pb=r.boxes.xyxy.cpu().numpy().astype(np.float32)
            ps=r.boxes.conf.cpu().numpy().astype(np.float32)
            pc=r.boxes.cls.cpu().numpy().astype(np.int64)
        else:
            pb=np.zeros((0,4),dtype=np.float32)
            ps=np.zeros((0,),dtype=np.float32)
            pc=np.zeros((0,),dtype=np.int64)
        records.append({'image_path':str(img_path), 'pred_boxes':pb,'pred_scores':ps,
                        'pred_classes':pc,'gt_boxes':gt['boxes'],'gt_classes':gt['classes']})
    del results,m
    gc.collect(); torch.cuda.empty_cache()
    assert len(records)==expected, f'Jumlah prediksi beda: {len(records)} vs {expected}'
    return records

def evaluate_validation(checkpoint, label):
    records=generate_records_for_checkpoint(checkpoint,'valid',EVAL_MAX_VALID_IMAGES)
    scores=[]
    for conf in CONF_GRID:
        summary, _=evaluate_f1(records,float(conf),MATCH_IOU)
        scores.append(summary)
    table=pd.DataFrame(scores).sort_values(['macro_f1_supported','micro_f1'],ascending=False).reset_index(drop=True)
    best_conf=float(table.iloc[0]['conf_threshold'])
    best_metrics, best_by_class=evaluate_f1(records,best_conf,MATCH_IOU)
    print(f'{label}: best valid conf={best_conf:.2f} | macro-F1={best_metrics["macro_f1_supported"]:.5f}')
    return dict(conf=best_conf, metrics=best_metrics, per_class=best_by_class, sweep=table)

In [ ]:
# 11. Pilih confidence validation; bandingkan Macro F1, tidak menggunakan test untuk tuning
val_results=evaluate_validation(FINAL_PT,'YOLO26n FAST T4x2')
print('VAL Macro F1 @ IoU=.50 =',round(val_results['metrics']['macro_f1_supported']*100,2),'%')
print('Confidence terbaik validation =',val_results['conf'])
display(val_results['per_class'][['class_name','gt','precision','recall','f1']])
val_results['per_class'].to_csv(EVAL_ROOT/'fast_val_per_class.csv',index=False)
(EVAL_ROOT/'fast_val_threshold.json').write_text(json.dumps({'conf':float(val_results['conf'])},indent=2))

In [ ]:
# 12. Evaluasi test hanya sekali setelah konfigurasi final dipilih pada validation.
CONF=val_results['conf']
test_records=generate_records_for_checkpoint(FINAL_PT,'test')
test_summary,test_per_class=evaluate_f1(test_records,CONF,MATCH_IOU)
print('TEST Macro F1 @ IoU=.50 =',round(test_summary['macro_f1_supported']*100,2),'%')
display(test_per_class[['class_name','gt','f1']])
test_per_class.to_csv(EVAL_ROOT/'fast_test_per_class.csv',index=False)
res=YOLO(str(FINAL_PT)).val(data=str(DATA_YAML),split='test',imgsz=640,device=0,batch=16,workers=2,plots=False,verbose=False)
print('TEST mAP@50 =',round(res.box.map50*100,2),'%')

### Checklist sebelum submit
- Model untuk submission: `/kaggle/working/telepati_abf/exports/fast_t4x2/final_best.pt` (setelah tahap 2).
- Jika dua GPU lambat, ubah `FAST_DEVICE='0'`, `BATCH_STAGE1=16`, `BATCH_STAGE2=12` lalu uji run baru; jangan bandingkan kecepatan dengan batch yang berbeda tanpa menyebut perbedaannya.
- Selalu simpan **checkpoint ke penyimpanan permanen** melalui Save Version/download. Salinan dalam `/kaggle/working` saja tidak aman terhadap runtime yang terhapus.
- `paddy` dan `Rice-Leaf-Diseasee` dipetakan ke kelas `bacterial_leaf_blight` mengikuti notebook awal. Validasi manual mapping ini penting karena dapat menjadi sumber noise serius.
- Versi ini **tidak menggunakan zero-shot**; semua tahap memakai label train TELEPATI, dan test tidak dipakai untuk pemilihan model.